In [2]:
import xarray as xr
import glob
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import dask
import numpy as np
import os

In [5]:
cluster = PBSCluster(
    cores=1,
    memory='16GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='2:00:00'
)
cluster.scale(jobs=30)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.174:44405,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [9]:
test = xr.open_dataset('/gdex/data/d633000/e5.oper.an.pl/194001/e5.oper.an.pl.128_133_q.ll025sc.1940013000_1940013023.nc')
test

<xarray.Dataset> Size: 4GB
Dimensions:    (time: 24, level: 37, latitude: 721, longitude: 1440)
Coordinates:
  * time       (time) datetime64[ns] 192B 1940-01-30 ... 1940-01-30T23:00:00
  * level      (level) float64 296B 1.0 2.0 3.0 5.0 ... 925.0 950.0 975.0 1e+03
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Data variables:
    Q          (time, level, latitude, longitude) float32 4GB ...
    utc_date   (time) int32 96B ...
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB 1 data to netC...
    NETCDF_VERSION:       4.8.1
    CONVERSION_PLATFORM:  Linux r5i0n21 4.12.14-95.51-default #1 SMP Fri Apr ...
    CONVERSION_DATE:      Thu Mar 16 20:33:45 MDT 2023
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Thu Mar 16 20:34:15 2023: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 5.0.3 (Homepage = http://n...

In [4]:
test.sel(level=1000).nbytes

99688624

In [7]:
years = np.arange(1940, 2026, 1)
years

array([1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1950,
       1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961,
       1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972,
       1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983,
       1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994,
       1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005,
       2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016,
       2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])

In [ ]:
path = '/gdex/data/d633000/e5.oper.an.pl/'
save_path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/Q/'

q_files = []
for year in years:
    save_path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/Q/'
    
    pattern = f"{path}{year}*/*133_q*.nc"
    year_files = sorted(glob.glob(pattern))
    Q_year = xr.open_mfdataset(year_files, combine='nested', concat_dim='time',
                               engine='h5netcdf', parallel=True,
                               coords='minimal', compat='override',
                               chunks={'time': 1440, 'latitude': -1, 'longitude': -1})
    Q_year = Q_year
    Q_crop = Q_year['Q'].sel(latitude=slice(28, 7.75), longitude=slice(271, 301), level=1000)

    save_path = os.path.join(save_path, f'cropped_Q_{year}.nc')
    Q_crop.to_netcdf(save_path, mode='w')
    Q_year.close()

In [5]:
# def crop_q_ds(ds):
#     ds = ds['Q']
#     ds_crop = ds.sel(latitude=slice(7.75, 28), longitude=slice(271, 301), level=1000)
#     return ds_crop

In [8]:
# Q = Q.chunk({'time': (Q.sizes['time'] / 24), 'latitude': -1, 'longitude': -1})
# Q

<xarray.DataArray 'Q' (time: 753888, latitude: 82, longitude: 121)> Size: 30GB
dask.array<rechunk-merge, shape=(753888, 82, 121), dtype=float32, chunksize=(31412, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
    level      float64 8B 1e+03
Attributes: (12/14)
    long_name:                                          Specific humidity
    short_name:                                         q
    units:                                              kg kg**-1
    original_format:                                    WMO GRIB 1 with ECMWF...
    ecmwf_local_table:                                  128
    ecmwf_parameter:                                    133
    ...                                                 ...
    grid_specification:                                 0.25 degree x 0.25 de...
    rda_dataset:                                        ds633.0
    rda_dataset_url:                                    https:/rda.ucar.edu/d...
    rda_dataset_doi:                                    DOI: 10.5065/BH6N-5N20
    rda_dataset_group:                                  ERA5 atmospheric pres...
    QuantizeGranularBitGroomNumberOfSignificantDigits:  7

In [9]:
# Q.to_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/Q', mode='w')

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:3415: UserWarning: Sending large graph of size 596.98 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
Task exception was never retrieved
future: <Task finished name='Task-700204' coro=<Client._gather.<locals>.wait() done, defined at /glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:2367> exce

In [ ]:
test = xr.open_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/Q')
test

In [ ]:
client.shutdown()